# Week 18, Day 2: Binary search

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w018d2.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
n = 1_000_000
steps = 0
while n > 0:
    n //= 2
    steps += 1
print(steps)

In [ ]:
def binary_search(items, target):
    lo, hi = 0, len(items) - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        if items[mid] == target:
            return mid
        if items[mid] < target:
            lo = mid + 1
        else:
            hi = mid - 1
    return -1

data = [2, 5, 8, 12, 16, 23, 38, 41, 56, 72, 85, 91]
print(binary_search(data, 23), binary_search(data, 2), binary_search(data, 91), binary_search(data, 40))

*Interactive figure on the website: The highlighted part is where the target could still be. Each step throws half of it away..*

> **Common mistake.** Binary search only works on **sorted** data. On unsorted data it doesn't crash; it gives wrong answers, because "the target must be to the right" is no longer true. Before using it, be sure the list is sorted, by the same order you're searching with.

In [ ]:
def broken(items, target):
    lo, hi = 0, len(items) - 1
    passes = 0
    while lo <= hi and passes < 20:     # a guard, so the demonstration stops
        passes += 1
        mid = (lo + hi) // 2
        if items[mid] == target:
            return mid
        if items[mid] < target:
            lo = mid                    # bug: should be mid + 1
        else:
            hi = mid - 1
    return f"gave up after {passes} passes"

print(broken([10, 20], 20))
print(broken([10, 20], 15))

In [ ]:
from bisect import bisect_left, bisect_right, insort

marks = [35, 48, 48, 48, 61, 72, 90]
print(bisect_left(marks, 48), bisect_right(marks, 48))   # first 48, just past the last
print(bisect_right(marks, 48) - bisect_left(marks, 48))  # how many 48s: O(log n)
print(bisect_left(marks, 50))                             # where 50 would go

i = bisect_left(marks, 61)
print(i < len(marks) and marks[i] == 61)                  # is 61 there?

insort(marks, 55)
print(marks)

In [ ]:
from bisect import bisect_right

def grade(mark):
    cutoffs = [40, 50, 60, 70]         # sorted lower edges of each band
    letters = "FDCBA"
    return letters[bisect_right(cutoffs, mark)]

print([grade(m) for m in (12, 40, 49, 50, 65, 70, 99)])

In [ ]:
def isqrt(n):
    lo, hi = 0, n
    while lo < hi:
        mid = (lo + hi + 1) // 2      # round up, so lo = mid always makes progress
        if mid * mid <= n:
            lo = mid                   # mid works: the answer is mid or bigger
        else:
            hi = mid - 1               # mid is too big
    return lo

print(isqrt(10), isqrt(16), isqrt(10 ** 30))

> **Key idea.** Binary search: on sorted data, keep `lo` and `hi` around the part where the target could be, and halve it each time. O(log n). Decide what the bounds mean, and move past `mid` after checking it.

### Exercise: Write it yourself

Write `binary_search(items, target)` with a `while` loop (no `bisect`, no recursion) that returns the position of `target` in a sorted list, or -1.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_values():\n    \"several\"\n    data = [2, 5, 8, 12, 16, 23, 38, 41, 56, 72, 85, 91]\n    for i, x in enumerate(data):\n        assert binary_search(data, x) == i\n    for x in (0, 3, 40, 100):\n        assert binary_search(data, x) == -1\n    assert binary_search([], 5) == -1 and binary_search([5], 5) == 0\n    big = list(range(0, 2_000_000, 2))\n    assert binary_search(big, 1_234_568) == 617_284 and binary_search(big, 7) == -1\n    assert \"bisect\" not in source and \"binary_search(\" not in source.split(\"def binary_search\", 1)[1]")

### Exercise: Count the comparisons

Write `search_steps(items, target)` that does a binary search and returns how many times it looked at a middle item. Then set `worst_million` to the most steps it takes for any target in `list(range(1_000_000))` (you may work it out, or try every target).

In [ ]:
worst_million = ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    data = list(range(15))\n    assert search_steps(data, 7) == 1 and search_steps(data, 3) == 2 and search_steps(data, 0) == 4\n    assert search_steps(data, 99) == 4 and search_steps([], 1) == 0\n    assert worst_million == 20")

### Exercise: First and last

Write `first_position(items, target)` and `last_position(items, target)` for a sorted list that may contain repeats, each O(log n) and each returning -1 if the target isn't there. Write them with your own loops, not `bisect`. Then `count(items, target)` using the two.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("from bisect import bisect_left, bisect_right\ndef test_values():\n    \"several\"\n    data = [1, 2, 2, 2, 3, 5, 5, 8]\n    assert first_position(data, 2) == 1 and last_position(data, 2) == 3\n    assert first_position(data, 5) == 5 and last_position(data, 5) == 6\n    assert first_position(data, 4) == -1 and last_position(data, 9) == -1\n    assert count(data, 2) == 3 and count(data, 4) == 0 and count([], 1) == 0\n    big = sorted([i // 7 for i in range(700_000)])\n    assert first_position(big, 5000) == 35000 and last_position(big, 5000) == 35006 and count(big, 99_999) == 7\n    assert \"bisect\" not in source")

### Exercise: How many in a range?

A sorted list of house prices is in `PRICES`. Using `bisect`, write `in_range(prices, low, high)` returning how many prices are from `low` to `high` inclusive, and `nearest(prices, x)` returning the price closest to `x` (the lower one if two are equally close). Both must be O(log n). The list has at least one price.

In [ ]:
from bisect import bisect_left, bisect_right

PRICES = [150_000, 182_500, 199_950, 210_000, 210_000, 245_000, 310_000, 395_000]

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_values():\n    \"several\"\n    assert in_range(PRICES, 200_000, 250_000) == 3\n    assert in_range(PRICES, 210_000, 210_000) == 2\n    assert in_range(PRICES, 400_000, 500_000) == 0 and in_range(PRICES, 0, 10 ** 9) == 8\n    assert nearest(PRICES, 201_000) == 199_950\n    assert nearest(PRICES, 206_000) == 210_000\n    assert nearest(PRICES, 1) == 150_000 and nearest(PRICES, 10 ** 9) == 395_000\n    assert nearest(PRICES, 205_025) == 210_000 and nearest(PRICES, 204_975) == 199_950\n    rng = random.Random(8)\n    big = sorted(rng.randint(0, 10 ** 9) for _ in range(200_000))\n    for x in (5, 123_456_789, 999_999_999):\n        assert nearest(big, x) == min(big, key=lambda p: (abs(p - x), p))")

### Exercise: Search for the answer

Write `icbrt(n)` returning the largest whole number `r` with `r * r * r <= n`, for any whole number `n` of 0 or more, using binary search on the answer (no `**` with fractions, no `math`). It must be quick for numbers with 60 digits.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    assert [icbrt(n) for n in (0, 1, 7, 8, 26, 27, 28, 1000)] == [0, 1, 1, 2, 2, 3, 3, 10]\n    big = 123456789123456789 ** 3\n    assert icbrt(big) == 123456789123456789 and icbrt(big - 1) == 123456789123456788\n    assert icbrt(10 ** 60) == 10 ** 20\n    assert \"math\" not in source and \"** (1\" not in source and \"**(1\" not in source")

### Bug hunt: Misses the end

This search can't find a target that is the only item left to check. For example, it can't find 9 in `[1, 3, 5, 7, 9]`. Fix the one character that's wrong.

In [ ]:
def binary_search(items, target):
    lo, hi = 0, len(items) - 1
    while lo < hi:
        mid = (lo + hi) // 2
        if items[mid] == target:
            return mid
        if items[mid] < target:
            lo = mid + 1
        else:
            hi = mid - 1
    return -1

data = [1, 3, 5, 7, 9]
print([binary_search(data, x) for x in data])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"all found\"\n    expect_output(\"[0, 1, 2, 3, 4]\")")

### Bug hunt: Unsorted

This program asks whether some names are in a list using binary search, and gets two answers wrong. The search itself is correct. Fix the program with one change.

In [ ]:
from bisect import bisect_left

names = ["Mo", "Ada", "Zoe", "Ben", "Kit"]

def contains(items, x):
    i = bisect_left(items, x)
    return i < len(items) and items[i] == x

print([contains(names, n) for n in ["Ada", "Zoe", "Kit", "Eve"]])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"right answers\"\n    expect_output(\"[True, True, True, False]\")")

### Puzzle: Binary search

Arrange the lines into a correct iterative binary search. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them

Work these out without running them. `from bisect import bisect_left, bisect_right` has been done.

In [ ]:
a = ___   # bisect_left([10, 20, 20, 30], 20)
b = ___   # bisect_right([10, 20, 20, 30], 20)
c = ___   # bisect_left([10, 20, 20, 30], 25)
d = ___   # the most comparisons binary search needs for 1,000 items

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert a == 1 and b == 3 and c == 3 and d == 10")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: the smallest van

A removal firm must move boxes in a fixed order over `days` days. Each day it loads the next boxes in order into one van, until adding another box would go over the van's capacity. Write:

- `days_needed(weights, capacity)`: how many days it takes with a van of that capacity (every box fits on its own, so `capacity` is at least the heaviest box);
- `smallest_van(weights, days)`: the smallest whole-number capacity that moves all the boxes in at most `days` days.

`smallest_van` must use binary search on the answer: the capacity is somewhere between the heaviest box and the total weight, and a bigger van never needs more days. It must handle 100,000 boxes.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_days():\n    \"days_needed\"\n    w = [3, 2, 2, 4, 1, 4]\n    assert days_needed(w, 4) == 5 and days_needed(w, 6) == 3 and days_needed(w, 16) == 1\ndef test_small():\n    \"smallest_van\"\n    w = [3, 2, 2, 4, 1, 4]\n    assert smallest_van(w, 3) == 6\n    assert smallest_van(w, 1) == 16 and smallest_van(w, 6) == 4\n    assert smallest_van([1, 2, 3, 4, 5, 6, 7, 8, 9, 10], 5) == 15\n    assert smallest_van([7], 3) == 7\ndef test_big():\n    \"big\"\n    rng = random.Random(6)\n    w = [rng.randint(1, 500) for _ in range(100_000)]\n    v = smallest_van(w, 40)\n    assert days_needed(w, v) <= 40 and days_needed(w, v - 1) > 40\n    assert v == 626266")

## Warm-up

20 halvings take a million down to nothing. That's the most comparisons binary search ever needs for a million sorted items, against up to a million for linear search.

## Halving with a loop

Keep two positions, `lo` and `hi`, marking the part of the list where the target could still be. At the start that's the whole list. Each time round, look at the middle item of that part:

- if it **is** the target, you've found it;
- if it's **less** than the target, the target can only be to its right: move `lo` to `mid + 1`;
- if it's **more**, the target can only be to its left: move `hi` to `mid - 1`.

When `lo` passes `hi`, there's nothing left to search, and the target isn't there.

Step through it, and try a target that isn't in the list:

Each pass halves the part still in play, so after k passes about n/2ᵏ items are left. It finishes when that's below 1, after about log₂ n passes: **O(log n)** time, O(1) extra space (the recursive version in Week 16 used O(log n) stack space).

| Items | Linear search worst case | Binary search worst case |
|---|---|---|
| 1,000 | 1,000 | 10 |
| 1,000,000 | 1,000,000 | 20 |
| 1,000,000,000 | 1,000,000,000 | 30 |

## Getting the boundaries right

Binary search is famous for being easy to describe and easy to get wrong. Jon Bentley reported that when he asked professional programmers to write it, most versions had bugs. The trick is to decide exactly what `lo` and `hi` mean and keep that true:

**The invariant:** if the target is in the list at all, it's somewhere in `items[lo]` to `items[hi]`, inclusive.

Everything follows from that:

- Start with `lo = 0`, `hi = len(items) - 1`: the whole list.
- Loop `while lo <= hi`: while the part is not empty. (With `<`, a part of one item would never be checked.)
- After checking `mid`, move **past** it: `lo = mid + 1` or `hi = mid - 1`. (With `lo = mid`, a part of two items can loop for ever, because `mid` rounds down to `lo` again.)

Each pass either returns or makes the part strictly smaller, so the loop always ends.

With `lo = 0, hi = 1`, `mid` is 0 every time, and `lo = mid` changes nothing. With the guard removed, the loop would never end.

(In languages with fixed-size integers, `(lo + hi) // 2` can overflow on huge arrays, a bug found in Java's own library in 2006. Python's integers don't overflow, so the simple form is safe here.)

## The bisect module

Python's `bisect` module has binary search built in, in a slightly different and very useful form: instead of "where is it?", it answers "**where would it go?**"

- `bisect_left(items, x)`: the first position where `x` could be inserted to keep the list sorted. If `x` is in the list, that's the position of its **first** copy.
- `bisect_right(items, x)` (also called `bisect`): the position just **after** the last copy of `x`.
- `insort(items, x)`: inserts `x` in the right place (O(log n) to find the place, but O(n) to shift the items along).

Counting the items in a **range** of values is just two bisects: the number of marks from 40 to 70 is `bisect_right(marks, 70) - bisect_left(marks, 40)`. That's O(log n), however many items are in the range.

A classic use is turning a number into a band, such as a mark into a grade:

## Searching for an answer

Binary search works on anything where a yes/no question changes from "no" to "yes" exactly once as a number grows. Then you can find the point where it changes without checking every number.

The whole-number square root of n is the biggest `r` with `r * r <= n`. As `r` grows, `r * r <= n` goes from true to false once, so binary search finds it:

Here the invariant is "the answer is between `lo` and `hi`", and the loop stops when they meet. Notice the rounding up: since this version moves `lo` to `mid` (not past it), `mid` must never equal `lo`, or it would loop for ever.

This idea, **binary search on the answer**, solves many problems of the form "what's the smallest x that works?" when checking whether a given x works is easy. The daily challenge is one.

## Sort once, search many times

Binary search needs sorted data, and sorting costs O(n log n). So:

- **One search:** linear search, O(n). Sorting first would cost more than it saves.
- **Many searches (k of them):** sort once, then binary search: O(n log n + k log n), against O(k × n) for k linear searches.
- **Many "is it there?" checks:** a set is O(1) each. Binary search wins when you need **order**: the nearest value, everything in a range, where something would go.

## Practice

## Quiz

## Daily challenge

## Before you go

Play the number-guessing game with someone: they think of a number from 1 to 1,000, and you find it in at most 10 guesses by always guessing the middle. Then think of one everyday decision that's a binary search on the answer (the right oven temperature, the right volume). Tomorrow: sorting, starting with selection sort.